# Imports

In [1]:
import os
import re
import jsonlines
import pandas as pd
import pyterrier as pt
from bs4 import BeautifulSoup
from langdetect import detect, DetectorFactory

DetectorFactory.seed = 42

if not pt.started():
    pt.init()

/var/folders/40/7dldznv1543g_gft348zpllc0000gn/T/ipykernel_66398/1554622767.py:11: DeprecationWarning: Call to deprecated function (or staticmethod) started. (use pt.java.started() instead) -- Deprecated since version 0.11.0.
  if not pt.started():
Java started and loaded: pyterrier.java, pyterrier.terrier.java [version=5.11 (build: craig.macdonald 2025-01-13 21:29), helper_version=0.0.8]
/var/folders/40/7dldznv1543g_gft348zpllc0000gn/T/ipykernel_66398/1554622767.py:12: DeprecationWarning: Call to deprecated method pt.init(). Deprecated since version 0.11.0.
java is now started automatically with default settings. To force initialisation early, run:
pt.java.init() # optional, forces java initialisation
  pt.init()


# Load Data

In [2]:
class CleanDataLoader:
    def __init__(self, file_path):
        self.file_path = file_path
        self._df = None

    def load(self):
        if self._df:
            return self._df
        self._load_docs()
        return self._df

    def _load_docs(self):
        docs = []
        with jsonlines.open(self.file_path) as reader:
            for obj in reader:
                doc_id = obj["id"]
                content = f"{obj['title']} {obj['keywords']} {obj['text']}"
                docs.append({"docno": doc_id, "text": content})
        self._df = pd.DataFrame(docs)


data_loader = CleanDataLoader("data/corpus.jsonl")

In [3]:
def is_english(text):
    try:
        return detect(text) == "en"
    except:
        return False


def clean_text(text):
    text = BeautifulSoup(text, "html.parser").get_text()
    text = text.lower()
    text = re.sub(r"[^a-z\s]", " ", text)  # Keep only letters
    text = re.sub(r"\s+", " ", text).strip()
    return text


class CleanDataLoader:
    def __init__(self, file_path):
        self.file_path = file_path
        self._df = None

    def load(self):
        if self._df is not None:
            return self._df
        self._load_docs()
        return self._df

    def _load_docs(self):
        docs = []
        with jsonlines.open(self.file_path) as reader:
            for obj in reader:
                full_text = f"{obj['title']} {obj['keywords']} {obj['text']}"
                if is_english(full_text):
                    cleaned = clean_text(full_text)
                    docs.append({"docno": obj["id"], "text": cleaned})
        self._df = pd.DataFrame(docs)


clean_data_loader = CleanDataLoader("data/corpus.jsonl")

# Pre-processing and Indexing

## Raw data + Simple Indexer

In [4]:
index_path = "./index_raw"

if os.path.exists(index_path) and os.listdir(index_path):
    raw_index = pt.IndexFactory.of(index_path)
else:
    df = data_loader.load()
    records = df.to_dict(orient="records")
    indexer = pt.IterDictIndexer(index_path)
    indexref = indexer.index(records)
    raw_index = pt.IndexFactory.of(indexref)

## Clean data + Simple Indexer

In [ ]:
index_path = "./index_clean"

if os.path.exists(index_path) and os.listdir(index_path):
    raw_index = pt.IndexFactory.of(index_path)
else:
    df = clean_data_loader.load()
    records = df.to_dict(orient="records")
    indexer = pt.IterDictIndexer(index_path)
    indexref = indexer.index(records)
    raw_index = pt.IndexFactory.of(indexref)

## Load Queries

In [ ]:
queries_df = pd.read_csv("data/test_queries.csv")
queries_df = queries_df.rename(columns={"QueryId": "qid", "Query": "query"})
queries_df

,qid,query
0,2,roman architecture
1,4,france second world war normandy
2,6,d-day normandy invasion
3,8,eiffel
4,11,indian food
...,...,...
228,456,universities that are members of the sec confe...
229,457,sponsors of the mancuso quilt festivals
230,460,companies that john hennessey serves on the bo...
231,463,professional sports teams in philadelphia


# Retrieve Documents

## Kaggle Submission Helper

In [ ]:
outdir = "results"
os.makedirs(outdir, exist_ok=True)


def prepare_submission(results, output_file):
    output_file_path = os.path.join(outdir, output_file)
    submission = results.sort_values(["qid", "rank"])[["qid", "docno"]].copy()
    submission.columns = ["QueryId", "EntityId"]
    submission.to_csv(output_file_path, index=False)

## Raw Index

In [ ]:
def clean_query(text):
    text = str(text).lower()
    text = re.sub(r"[^\w\s]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text


queries_df["query"] = queries_df["query"].apply(clean_query)
queries_df

,qid,query
0,2,roman architecture
1,4,france second world war normandy
2,6,d day normandy invasion
3,8,eiffel
4,11,indian food
...,...,...
228,456,universities that are members of the sec confe...
229,457,sponsors of the mancuso quilt festivals
230,460,companies that john hennessey serves on the bo...
231,463,professional sports teams in philadelphia


## TF-IDF

In [ ]:
tf_idf = pt.terrier.Retriever(
    raw_index, wmodel="TF_IDF", num_results=100, metadata=["docno"]
)
tf_idf_results = tf_idf.transform(queries_df)
prepare_submission(tf_idf_results, "raw_index_tf_idf.csv")

## BM25

In [ ]:
bm25 = pt.terrier.Retriever(
    raw_index, wmodel="BM25", num_results=100, metadata=["docno"]
)
bm25_results = bm25.transform(queries_df)
prepare_submission(bm25_results, "raw_index_bm25.csv")

# Query Expansion

## TF-IDF

In [ ]:
tfidf_feedback = pt.terrier.Retriever(
    raw_index, wmodel="TF_IDF", num_results=10, metadata=["docno"]
)

qe = pt.rewrite.Bo1QueryExpansion(raw_index)

tfidf_final = pt.terrier.Retriever(
    raw_index, wmodel="TF_IDF", num_results=100, metadata=["docno"]
)

tfidf_qe_pipeline = tfidf_feedback >> qe >> tfidf_final

tfidf_expanded_results = tfidf_qe_pipeline.transform(queries_df)

prepare_submission(tfidf_expanded_results, "tfidf_expanded.csv")

## BM25

In [ ]:
bm25_feedback = pt.terrier.Retriever(
    raw_index, wmodel="BM25", num_results=10, metadata=["docno"]
)

qe = pt.rewrite.Bo1QueryExpansion(raw_index)

bm25_final = pt.terrier.Retriever(
    raw_index, wmodel="BM25", num_results=100, metadata=["docno"]
)

pipeline = bm25_feedback >> qe >> bm25_final

bm25_expanded_results = pipeline.transform(queries_df)

prepare_submission(bm25_expanded_results, "bm25_expanded.csv")